# Grab OpenStreetMap Data for Solar Panels and Arrays

## Import Libraries

In [1]:
# Import gmseus utilities functions
import gmseusUtils as gu
import osmUtils as ou

# Load config once (passed to util functions that need it)
config = gu.load_config()

ImportError: DLL load failed while importing _warp: The specified procedure could not be found.

## Set Required Paths

In [ ]:
# Set paths
wd = r'S:\Users\stidjaco\R_files\BigPanel'
pbfDir = r'F:\Data\Planet_OSM' # Local folder for large OSM PBF files (planet or Geofabrik extracts)

## Select OSM Extraction Method
- `'api'`: Download each region from the Overpass API via osmnx (original method; limited by Overpass throughput/rate limits)
- `'pbf'`: Bulk method. Use a local PBF (full planet or a Geofabrik extract), filter all solar features in one pass, then clip per region from that small extract. Same tags and the same downstream processing as `'api'`.

For `'pbf'`, set `pbfSource` to either an existing local PBF path (e.g., the planet file) or a Geofabrik region to download (`'north-america'` covers USA, Canada, Mexico, and Greenland; Central America/Caribbean countries are in `'central-america'`).

In [ ]:
# Select OSM extraction method: 'api' or 'pbf'
osmMethod = 'pbf'

# PBF source (only used if osmMethod == 'pbf'): a local .osm.pbf path, or a Geofabrik region name to download
pbfSource = 'north-america' # e.g., os.path.join(pbfDir, 'planet-251124.osm.pbf'), 'north-america', 'north-america/us', 'central-america'

## Build Bulk Solar Extract (PBF Method Only)
Downloads the Geofabrik extract if needed (~15+ GB for North America), then filters it to solar features once. Both steps are skipped if their outputs already exist; set `overwrite=True` to rebuild after downloading a newer PBF.

In [ ]:
import os

osmSolar = None
if osmMethod == 'pbf':
    # Get the PBF: use a local file if given, otherwise download from Geofabrik
    pbfPath = pbfSource if pbfSource.endswith('.pbf') else ou.downloadGeofabrikPBF(pbfSource, pbfDir)

    # Filter solar features from the PBF once (saved next to the other OSM downloads)
    pbfName = os.path.basename(pbfPath).replace('.osm.pbf', '')
    osmSolarPath = os.path.join(wd, r'Data\Downloaded\SolarDB\OSM\PBF', f'osmSolar_{pbfName}.gpkg')
    ou.extractSolarFromPBF(pbfPath, osmSolarPath)

    # Load once and reuse for all regions/countries below
    osmSolar = ou.loadSolarOSMExtract(osmSolarPath)

## Acquire Region-Wide Solar Array and Panel-Row OSM Data

In [ ]:
# Process solar OSM data for an individual region
ou.processSolarOSMData('Rhode Island', 'USA', wd, config, method=osmMethod, osmSolar=osmSolar)

## Acquire Contry-Wide Solar Array and Panel-Row OSM Data

In [ ]:
# Call the function to get OSM solar data for the entire country (CONUS)
ou.getCountrySolarOSMData('USA', wd, config, method=osmMethod, osmSolar=osmSolar)
ou.getCountrySolarOSMData('Canada', wd, config, method=osmMethod, osmSolar=osmSolar)
ou.getCountrySolarOSMData('Mexico', wd, config, method=osmMethod, osmSolar=osmSolar)

## Acquire OSM Solar Inverter Data (Points)
Feasibility check for inverter locations in OSM. There is no settled OSM tag for inverters, so features tagged with any of `power=inverter` (most common), `power=converter`, `power=inverter_station`, `substation=inverter`, or `transformer=inverter` are kept (see `ou.inverterTagFilter`). Note `power=converter` also includes HVDC converter stations.

Each inverter is matched to the nearest OSM solar array from the run above: `distArray` (m), `inArray` (touching/inside an array), and `nearArray` (within `invBufferDist`). Inverters often sit on pads just outside the mapped panel area, so use `nearArray`/`distArray` rather than strict containment. Requires the country solar run above first.

For `'pbf'`, the inverter extract must read every node in the PBF, so it takes longer than the solar extract (skipped if it already exists).

In [ ]:
# Distance (m) to count an inverter as belonging to an OSM solar array
invBufferDist = 100

# OSM inverter tags do not record the source (solar, battery, wind, rail...), so proximity to a mapped OSM solar array is the only solar evidence.
# False keeps all inverters (filter later on nearArray); True drops inverters farther than invBufferDist from an OSM solar array.
invSolarOnly = False

osmInverters = None
if osmMethod == 'pbf':
    # Filter inverter features from the PBF once (saved next to the solar extract)
    osmInvertersPath = os.path.join(wd, r'Data\Downloaded\SolarDB\OSM\PBF', f'osmInverters_{pbfName}.gpkg')
    ou.extractInvertersFromPBF(pbfPath, osmInvertersPath)
    osmInverters = ou.loadSolarOSMExtract(osmInvertersPath)

# Get inverters for every state, match to OSM solar arrays, and print a feasibility summary
invUSA = ou.getCountryInverterOSMData('USA', wd, config, method=osmMethod, osmInverters=osmInverters, bufferDist=invBufferDist, solarOnly=invSolarOnly)